# Inspect InsightQMC Checkpoint

下面这个代码块会读取 `outputs/carbon_spinblock_test_001/checkpoints/last.pkl`，查看 checkpoint 里保存了哪些内容，包括 `params`、`data`、优化器状态和 MKAN 静态状态。

In [14]:
from pathlib import Path
import pickle
import sys
from collections.abc import Mapping

import numpy as np

# Make the notebook work whether Jupyter is launched from InsightQMC/ or its parent directory.
project_root = Path.cwd()
if not (project_root / "networks.py").exists() and (project_root / "InsightQMC" / "networks.py").exists():
    project_root = project_root / "InsightQMC"
sys.path.insert(0, str(project_root))

# ckpt_path = project_root / "outputs" / "carbon_spinblock_test_001" / "checkpoints" / "last.pkl"
# ckpt_path = '/vepfs-mlp2/c20250516/250504030/jing/InsightQMC/outputs/H_LZW6021452/checkpoints/last.pkl'
# ckpt_path = '/vepfs-mlp2/c20250516/250504030/jing/InsightQMC/outputs/Li_LZW6021621/checkpoints/last.pkl'
# ckpt_path = '/vepfs-mlp2/c20250516/250504030/jing/InsightQMC/outputs/carbon_spinblock_test_LZW5182100/checkpoints/last.pkl'
ckpt_path = '/vepfs-mlp2/c20250516/250504030/jing/InsightQMC/outputs/C_LZW6022246/checkpoints/last.pkl'
with open(ckpt_path, "rb") as f:
    ckpt = pickle.load(f)

print("checkpoint path:", ckpt_path)
print("top-level keys:", list(ckpt.keys()))
print("stage:", ckpt.get("stage"))
print("step:", ckpt.get("step"))

def short_leaf(x):
    shape = getattr(x, "shape", None)
    dtype = getattr(x, "dtype", None)
    if shape is not None:
        return f"{type(x).__name__}(shape={tuple(shape)}, dtype={dtype})"
    return f"{type(x).__name__}: {x!r}"

def summarize_tree(x, name="root", max_depth=4, depth=0):
    indent = "  " * depth
    if depth > max_depth:
        print(f"{indent}{name}: ...")
        return
    if isinstance(x, Mapping):
        print(f"{indent}{name}: dict[{len(x)}]")
        for key, value in x.items():
            summarize_tree(value, str(key), max_depth=max_depth, depth=depth + 1)
    elif isinstance(x, (list, tuple)):
        print(f"{indent}{name}: {type(x).__name__}[{len(x)}]")
        for i, value in enumerate(x[:8]):
            summarize_tree(value, f"[{i}]", max_depth=max_depth, depth=depth + 1)
        if len(x) > 8:
            print(f"{indent}  ... {len(x) - 8} more")
    elif hasattr(x, "__dict__") and not hasattr(x, "shape"):
        print(f"{indent}{name}: {type(x).__name__}")
        for key, value in vars(x).items():
            summarize_tree(value, key, max_depth=max_depth, depth=depth + 1)
    else:
        print(f"{indent}{name}: {short_leaf(x)}")

print("\n== data ==")
summarize_tree(ckpt["data"], "data", max_depth=3)

print("\n== params ==")
summarize_tree(ckpt["params"], "params", max_depth=5)

print("\n== optimizer states ==")
summarize_tree(ckpt.get("pretrain_opt_state"), "pretrain_opt_state", max_depth=3)
summarize_tree(ckpt.get("train_opt_state"), "train_opt_state", max_depth=3)

print("\n== mkan_static_state ==")
summarize_tree(ckpt.get("mkan_static_state"), "mkan_static_state", max_depth=4)


checkpoint path: /vepfs-mlp2/c20250516/250504030/jing/InsightQMC/outputs/C_LZW6022246/checkpoints/last.pkl
top-level keys: ['stage', 'step', 'params', 'data', 'key', 'pretrain_opt_state', 'train_opt_state', 'mkan_static_state']
stage: train
step: 100000

== data ==
data: dict[4]
  positions: ArrayImpl(shape=(2048, 18), dtype=float32)
  spins: ArrayImpl(shape=(1, 6), dtype=int32)
  atoms: ArrayImpl(shape=(1, 3), dtype=float32)
  charges: ArrayImpl(shape=(1,), dtype=float32)

== params ==
params: dict[3]
  envelope: list[2]
    [0]: dict[2]
      c_basis: ArrayImpl(shape=(1, 8, 4), dtype=float32)
      sigma: ArrayImpl(shape=(1, 4), dtype=float32)
    [1]: dict[2]
      c_basis: ArrayImpl(shape=(1, 8, 2), dtype=float32)
      sigma: ArrayImpl(shape=(1, 2), dtype=float32)
  jastrow_ee: dict[4]
    ee_anti: ArrayImpl(shape=(1,), dtype=float32)
    ee_anti_coeff: ArrayImpl(shape=(4,), dtype=float32)
    ee_par: ArrayImpl(shape=(1,), dtype=float32)
    ee_par_coeff: ArrayImpl(shape=(4,), dty

In [15]:
# 如果只想看 MKAN 参数的所有叶子 shape，可以运行这一格。
try:
    import jax
    mkan_params = ckpt["params"]["mkan"]
    leaves = jax.tree_util.tree_leaves(mkan_params)
    print("number of MKAN parameter leaves:", len(leaves))
    print("total MKAN scalar parameters:", sum(int(np.prod(leaf.shape)) for leaf in leaves if hasattr(leaf, "shape")))
    for i, leaf in enumerate(leaves):
        print(i, short_leaf(leaf))
except Exception as exc:
    print("Could not summarize MKAN leaves:", repr(exc))


number of MKAN parameter leaves: 16
total MKAN scalar parameters: 3980
0 ArrayImpl(shape=(16,), dtype=float32)
1 ArrayImpl(shape=(64, 13), dtype=float32)
2 ArrayImpl(shape=(16, 4), dtype=float32)
3 ArrayImpl(shape=(16, 4), dtype=float32)
4 ArrayImpl(shape=(12,), dtype=float32)
5 ArrayImpl(shape=(192, 13), dtype=float32)
6 ArrayImpl(shape=(12, 16), dtype=float32)
7 ArrayImpl(shape=(12, 16), dtype=float32)
8 ArrayImpl(shape=(16,), dtype=float32)
9 ArrayImpl(shape=(12,), dtype=float32)
10 ArrayImpl(shape=(16,), dtype=float32)
11 ArrayImpl(shape=(12,), dtype=float32)
12 ArrayImpl(shape=(16,), dtype=float32)
13 ArrayImpl(shape=(12,), dtype=float32)
14 ArrayImpl(shape=(16,), dtype=float32)
15 ArrayImpl(shape=(12,), dtype=float32)
